In [2]:
%pylab inline
import torch 
import torch.nn as nn 
import torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from keras.datasets import mnist
import numpy as np
from tqdm import trange

Populating the interactive namespace from numpy and matplotlib


In [118]:
# Load Data 
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Learning Params 
epochs = 500
batch_size = 400 
shuffler = np.random.randint(0, x_train.shape[0], size=batch_size)


In [84]:
# GAN Models 

# Neurons 
image_size = 784 
latent_size = 64 
h1_size = 128
h2_size = 256  

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.L1 = nn.Linear(latent_size, h1_size)
        self.L2 = nn.Linear(h1_size, h2_size)
        self.L3 = nn.Linear(h2_size, image_size)
    
    def forward(self, x):
        x = self.L1(x)
        x = F.relu(x)
        x = self.L2(x)
        x = F.relu(x)
        x = self.L3(x)
        return x 

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.L1 = nn.Linear(image_size, h2_size)
        self.L2 = nn.Linear(h2_size, h1_size)
        self.L3 = nn.Linear(h1_size, latent_size)
        self.L4 = nn.Linear(latent_size, 1)
    
    def forward(self, x):
        x = self.L1(x)
        x = F.relu(x)
        x = self.L2(x)
        x = F.relu(x)
        x = self.L3(x)
        x = F.relu(x)
        x = self.L4(x)
        x = F.sigmoid(x)
        return x 




In [101]:
# Models 
g = Generator().to(device)
d = Discriminator().to(device)

# Optimization And Loss
g_optim = torch.optim.Adam(g.parameters())
d_optim = torch.optim.Adam(d.parameters())
loss_function = nn.BCELoss()

# Generate Fakes and Reals 
fakes = {"X": torch.randn(batch_size, latent_size).to(device),
  "Y": torch.zeros(batch_size, 1).to(device)}
reals = {"X": torch.tensor(x_train[shuffler].reshape(-1, 28*28)).to(device).float()/255.0,  "Y": torch.ones(batch_size, 1).to(device)}




In [114]:
# Train Discriminator 
def d_train():
    # Reals 
    real_outputs = d(reals["X"])
    real_loss = loss_function(real_outputs, reals["Y"])
    
    # Fakes
    fake_images = g(fakes["X"])
    fake_outputs = d(fake_images)
    fake_loss = loss_function(fake_outputs, fakes["Y"])

    # Update weights 
    d_loss = fake_loss + real_loss 
    d_optim.zero_grad()
    d_loss.backward()
    d_optim.step()
    return d_loss

# Train Generator 
def g_train():
    # Fake Forward 
    fake_images = g(fakes["X"])
    fake_outputs = d(fake_images)
    g_loss = loss_function(fake_outputs, reals["Y"])

    # Update Weights 
    g_optim.zero_grad()
    g_loss.backward()
    g_optim.step()
    return g_loss 



In [120]:
# TODO: Write training functions as one loop 
for _ in (t:= trange(epochs)):
    d_train()
    g_train()
    t.set_description(f"d_loss = {d_train()}, g_loss = {g_train()}")


d_loss = 1.5095479488372803, g_loss = 75.95154571533203: 100%|██████████| 700/700 [00:08<00:00, 78.84it/s]
